<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [ ]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.7 MB/s eta 0:00:00


# **3) Refinería Melrose**

***Restricciones críticas:***

Calidad mínima de gasolina (≥ 9)

Calidad mínima de aceite (≥ 7)

Límites de venta

***Interpretación:***

Las restricciones de calidad limitan el uso de grados bajos

Se priorizan métodos que generan más grado 10

***Sensibilidad:***

Si baja el requisito de calidad → aumenta la utilidad

Si sube el precio de gasolina → se produce más gasolina

Si aumentan los costos de procesamiento → cambia la mezcla de métodos

**Métodos de procesamiento**

Cada barril de crudo puede procesarse así:

***Método 1:***

Costo: 3.4

Produce:

0.2 grado 6/ 0.2 grado 8/ 0.6 grado 10

***Método 2:***

Costo: 3

Produce:

0.3 grado 6/0.3 grado 8/0.4 grado 10

***Método 3:***

Costo: 2.6

Produce:

0.4 grado 6/0.4 grado 8/0.2 grado 10

🔹 Productos finales

Gasolina

Precio: 8

Calidad: ≥ 9

Máximo: 2000 barriles

Aceite combustible

Precio: 6

Calidad: ≥ 7

Máximo: 600 barriles

**Mejoras de calidad**

Grado 6 → grado 8

Costo: 1.3

Grado 8 → grado 10

Costo: 2

**Desecho**

Exceso se elimina

Costo: 0.20 por barril

In [ ]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    return {
        "Metodo1": round(ampl.get_variable("x1").value(), 2),
        "Metodo2": round(ampl.get_variable("x2").value(), 2),
        "Metodo3": round(ampl.get_variable("x3").value(), 2),
        "Gasolina": round(ampl.get_variable("g").value(), 2),
        "Aceite": round(ampl.get_variable("f").value(), 2),
        "Utilidad": round(ampl.get_objective("Utilidad").value(), 2),
    }


@timed
def solve_refineria_problem():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        # VARIABLES
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;

        var g >= 0;   # gasolina
        var f >= 0;   # aceite

        # FUNCION OBJETIVO
        maximize Utilidad:
            8*g + 6*f
            - (3.4*x1 + 3*x2 + 2.6*x3);

        # PRODUCCION DE GRADOS (disponibles)
        # grado 6 total
        var G6 >= 0;
        var G8 >= 0;
        var G10 >= 0;

        s1: G6 = 0.2*x1 + 0.3*x2 + 0.4*x3;
        s2: G8 = 0.2*x1 + 0.3*x2 + 0.4*x3;
        s3: G10 = 0.6*x1 + 0.4*x2 + 0.2*x3;

        # USO DE PRODUCTOS
        s4: g + f <= G6 + G8 + G10;

        # CALIDAD PROMEDIO

        # gasolina ≥ 9
        s5:
        6*(G6) + 8*(G8) + 10*(G10) >= 9*g;

        # aceite ≥ 7
        s6:
        6*(G6) + 8*(G8) + 10*(G10) >= 7*f;

        # LIMITES DE VENTA
        s7: g <= 2000;
        s8: f <= 600;
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)


resultado, tiempo = solve_refineria_problem()

print("=== RESULTADOS EJERCICIO 3 ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 12840
1 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 3 ===
Solución -> {'Metodo1': 0.0, 'Metodo2': 0.0, 'Metodo3': 2600.0, 'Gasolina': 2000.0, 'Aceite': 600.0, 'Utilidad': 12840.0}
Tiempo   -> 6.582311 segundos
